In [1]:
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load one of the .mat files (Replace with your exact file name)
# Choose one with a low "drop" rate and high "classes" count.
file_name = 'feature_all_selected_drop1_antenna3_samples10_classes5.mat' 
mat_data = scipy.io.loadmat(file_name)

# 2. Automatically find the dynamic dictionary keys for your file
# (This handles the $A$ and $B$ naming confusion)
feature_key = [key for key in mat_data.keys() if key.startswith('features_')][0]
label_key = [key for key in mat_data.keys() if key.startswith('classes_')][0]

# 3. Extract Features (X) and Labels (y)
# The features are natively 120 x Samples. ML models need Samples x 120, so we transpose (.T)
X = mat_data[feature_key].T
y = mat_data[label_key].flatten()

print(f"Data Successfully Loaded!")
print(f"Total time samples: {X.shape[0]}")
print(f"Features per sample (Subcarriers): {X.shape[1]}")
print("-" * 30)

# 4. The IEEE Paper Implementation: Train the Random Forest
# Split data into 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Training the Random Forest model (this simulates the paper's core logic)...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 5. Test the model and see if we get near the paper's 98% accuracy
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on testing data: {accuracy * 100:.2f}%")
print("\nDetailed Breakdown (Can it tell the difference between 1 person and 4 people?):")
print(classification_report(y_test, y_pred))

Data Successfully Loaded!
Total time samples: 390
Features per sample (Subcarriers): 120
------------------------------
Training the Random Forest model (this simulates the paper's core logic)...
Model Accuracy on testing data: 89.74%

Detailed Breakdown (Can it tell the difference between 1 person and 4 people?):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        39
           1       1.00      1.00      1.00        22
           2       0.91      0.62      0.74        16
           3       0.62      0.88      0.73        17
           4       0.90      0.83      0.86        23

    accuracy                           0.90       117
   macro avg       0.89      0.87      0.87       117
weighted avg       0.91      0.90      0.90       117



In [3]:
from xgboost import XGBClassifier

print("\n--- Applying Project Modification: XGBoost ---")
print("Training the XGBoost model to resolve Class 2/3 confusion...")

# Initialize XGBoost. It automatically handles multi-class classification.
xgb_model = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    random_state=42,  
    eval_metric='mlogloss'
)

# Train the model
xgb_model.fit(X_train, y_train)

# Test the model
xgb_pred = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print(f"XGBoost Accuracy on testing data: {xgb_accuracy * 100:.2f}%")
print("\nDetailed Breakdown (Did we fix the 2 vs. 3 confusion?):")
print(classification_report(y_test, xgb_pred))


--- Applying Project Modification: XGBoost ---
Training the XGBoost model to resolve Class 2/3 confusion...
XGBoost Accuracy on testing data: 90.60%

Detailed Breakdown (Did we fix the 2 vs. 3 confusion?):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        39
           1       1.00      1.00      1.00        22
           2       0.72      0.81      0.76        16
           3       0.80      0.71      0.75        17
           4       0.87      0.87      0.87        23

    accuracy                           0.91       117
   macro avg       0.88      0.88      0.88       117
weighted avg       0.91      0.91      0.91       117



In [4]:
import time

print("\n--- Running Adaptive Wi-Fi Manager Simulation ---")
print("Simulating real-time router adjustments based on XGBoost predictions...\n")

# Let's simulate the router reading the first 10 seconds of your test data
for i, predicted_occupancy in enumerate(xgb_pred[:10]):
    print(f"Time {i}s | Predicted Occupants: {predicted_occupancy}")
    
    if predicted_occupancy == 0:
        print("   -> ACTION: Eco-Mode Activated. Reducing Tx power to 10% to save energy.\n")
        
    elif predicted_occupancy == 1:
        print("   -> ACTION: Precision Mode. Activating single-user beamforming for maximum throughput.\n")
        
    elif predicted_occupancy == 2:
        print("   -> ACTION: Standard Mode. Splitting bandwidth across standard MIMO antennas.\n")
        
    elif predicted_occupancy >= 3:
        print("   -> ACTION: Crowd Mode. Activating MU-MIMO (Multi-User) spatial multiplexing to prevent network congestion.\n")
        
    # Pause for half a second so it looks like a real-time terminal output
    time.sleep(0.5)


--- Running Adaptive Wi-Fi Manager Simulation ---
Simulating real-time router adjustments based on XGBoost predictions...

Time 0s | Predicted Occupants: 0
   -> ACTION: Eco-Mode Activated. Reducing Tx power to 10% to save energy.

Time 1s | Predicted Occupants: 0
   -> ACTION: Eco-Mode Activated. Reducing Tx power to 10% to save energy.

Time 2s | Predicted Occupants: 0
   -> ACTION: Eco-Mode Activated. Reducing Tx power to 10% to save energy.

Time 3s | Predicted Occupants: 3
   -> ACTION: Crowd Mode. Activating MU-MIMO (Multi-User) spatial multiplexing to prevent network congestion.

Time 4s | Predicted Occupants: 3
   -> ACTION: Crowd Mode. Activating MU-MIMO (Multi-User) spatial multiplexing to prevent network congestion.

Time 5s | Predicted Occupants: 4
   -> ACTION: Crowd Mode. Activating MU-MIMO (Multi-User) spatial multiplexing to prevent network congestion.

Time 6s | Predicted Occupants: 0
   -> ACTION: Eco-Mode Activated. Reducing Tx power to 10% to save energy.

Time 7s 